<a href="https://colab.research.google.com/github/WRiegs/Squidly/blob/main/SquidlyExampleColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Example notebook to run Squidly

This notebook will work either locally or on colab, however, note that on colab it can only run BLAST through squidly (locally you should be able to run both BLAST and Squidly). This is because there is a maximum amount of memory assigned. If you have the premium version you may be able to run both.



In [2]:
# install squidly and get the dataset
!pip install huggingface_hub
!pip install squidly
!squidly install
!wget https://raw.githubusercontent.com/WRiegs/Squidly/main/data/reviewed_sprot_08042025.csv.zip
!unzip reviewed_sprot_08042025.csv.zip
# install clustal for MSA
!apt-get update
!apt-get install -y clustalo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.3 MB/s eta 0:00:00
--------------------------------------------------------------------------------
                             Installing models... 	                             
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
 If this fails please see the github and follow the installation instructions.	 
--------------------------------------------------------------------------------
sh: 1: /usr/local/lib/python3.12/dist-packages/squidly/./install.sh: not found
Attempting to download HuggingFace snapshot into package dir: /usr/local/lib/python3.12/dist-packages/squidly
Fetching 22 fi

In [3]:
# install diamond BLAST (thanks to claude)
!apt-get update
!apt-get install -y build-essential cmake zlib1g-dev

# Clone the repository
!git clone https://github.com/bbuchfink/diamond.git

# Build DIAMOND
%cd diamond
!mkdir build
%cd build
!cmake ..
!make -j4

# Install
!make install

# Go back to root directory
%cd /content

# Verify installation
!diamond version

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest versi

## Create or upload your fasta file of interest

Here we just use two simple sequences that actually exist in our database.

In [4]:
with open('example.fasta', 'w+') as fout:
  fout.write('>A0A009IHW8\n\
  MSLEQKKGADIISKILQIQNSIGKTTSPSTLKTKLSEISRKEQENARIQSKLSDLQKKKI\n\
  DIDNKLLKEKQNLIKEEILERKKLEVLTKKQQKDEIEHQKKLKREIDAIKASTQYITDVS\n\
  ISSYNNTIPETEPEYDLFISHASEDKEDFVRPLAETLQQLGVNVWYDEFTLKVGDSLRQK\n\
  IDSGLRNSKYGTVVLSTDFIKKDWTNYELDGLVAREMNGHKMILPIWHKITKNDVLDYSP\n\
  NLADKVALNTSVNSIEEIAHQLADVILNR\n>A0A023I7E1\n\
  MRFQVIVAAATITMITSYIPGVASQSTSDGDDLFVPVSNFDPKSIFPEIKHPFEPMYANT\n\
  ENGKIVPTNSWISNLFYPSADNLAPTTPDPYTLRLLDGYGGNPGLTIRQPSAKVLGSYPP\n\
  TNDVPYTDAGYMINSVVVDLRLTSSEWSDVVPDRQVTDWDHLSANLRLSTPQDSNSYIDF\n\
  PIVRGMAYITANYNNLTPQFLSQHAIISVEADEKKSDDNTSTFSGRKFKITMNDDPTSTF\n\
  IIYSLGDKPLELRKQDNSNLVASKPYTGVIRVAKLPAPEFETLLDASRAVWPTGGDISAR\n\
  SDDNNGASYTIKWKTNSNEAPLLTYAYAHHLTSIDDSNVKRTDMTLQSATKGPMTALVGN\n\
  EWTLRETELSPVEWLPLQAAPNPTTINEIMTEINKDIASNYTQETAKEDNYFSGKGLQKF\n\
  AMLALILNKSDQTQLRNPELAQIALDKLKAAFLPYLQNEQADPFRYDTLYKGIVAKAGLP\n\
  TSMGGTDDLSAEFGHSYYSDHHYHQGYFVVTAAIIHHLDPTWNADRLKAWTEALIRDVNN\n\
  ANDGDEYFAAFRNWDWFAGHSWAGGIKPDGALDGRDQESVPESVNFYWGAKLWGLATGNT\n\
  PLTKLASLQLAVTKRTTYEYFWMLDGNKNRPENIVRNKVIGIYFEQKTDYTTYFGRFLEY\n\
  IHGIQQLPMTPELMEYIRTPEFVSQEWDEKLGAIAPTVQSPWAGVLYLNYAIINPAEAYP\n\
  ALRKVQMDDGQTRSYSLYLTATRPHFFRRSLLAALARHGSTRRPSLPSSGDDDKHEDGFL\n\
  LRFRRLNPFNLKHRIY\n')


## Run Squidly

Here we can run Squidly using the newly added CPU version!

Unfortunately, Squidly only just exceeds the RAM limits of the standard free runtime option (12.7GB as of Jan 2026). Fortunately Colab is currently providing free access to alternative runtime environments (subject to availability). The TPU options are great for running the CPU version, and provide 47GB ram.

If you have access to colab GPUs, you can remove the --cpu tag.

In [9]:
! squidly run example.fasta esm2_t36_3B_UR50D --cpu

--------------------------------------------------------------------------------
                             Starting squidly... 	                              
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
                               Running ensemble	                                
--------------------------------------------------------------------------------
/usr/local/lib/python3.12/dist-packages/squidly/models
--------------------------------------------------------------------------------
Running non-chunked command:	python /usr/local/lib/python3.12/dist-packages/squidly/squidly.py /content/squidly_input_fasta.fasta esm2_t36_3B_UR50D /usr/local/lib/python3.12/dist-packages/squidly/models/3B /content --cpu	
--------------------------------------------------------------------------------
Loaded 5 models for the ensemble
Read /content/squidly_input_fasta.fasta wi

Results are now in the folder (squidly_input_fasta_squidly_ensemble.csv).

Please feel free to post an issue if anything isn't clear!

---


## Run squidly (Blast)

Here you can run squidly's Blast mode.

This is the basic version, note by passing `--database` it will also run BLAST (actually it will only run BLAST on these two sequences since both get a match).

If you are running locally and want to run just with squidly remove the `--database reviewed_sprot_08042025.csv`.

In [ ]:
! squidly run example.fasta esm2_t36_3B_UR50D --database reviewed_sprot_08042025.csv

--------------------------------------------------------------------------------
                             Starting squidly... 	                              
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
        Running BLAST on the following DB: 	reviewed_sprot_08042025.csv	        
--------------------------------------------------------------------------------
diamond v2.1.17.171 (C) Max Planck Society for the Advancement of Science, Benjamin J. Buchfink, University of Tuebingen
Documentation, support and updates available at http://www.diamondsearch.org
Please cite: http://dx.doi.org/10.1038/s41592-021-01101-x Nature Methods (2021)

#CPU threads: 2
Scoring parameters: (Matrix=BLOSUM62 Lambda=0.267 K=0.041 Penalties=11/1)
Database input file: /content/squidly_database.fasta
Opening the database file...  [0.002s]
Loading sequences...  [0.568s]
Masking sequences...  

Results are now in the folder (squidly_blast.csv).

Please feel free to post an issue if anything isn't clear!